# Location Classifier

An end-to-end walkthrough of the `location_classifier` package: build a dataset,
engineer geodesy-aware features, train and compare models, inspect the errors,
and cluster the points without labels.

All logic lives in the importable package - this notebook only drives it, so
nothing here is trapped in a `.ipynb`.

Run `pip install -r requirements.txt` first.

In [ ]:
import sys
from pathlib import Path

# Make the package importable when the notebook is opened from the repo root.
sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import pandas as pd

from location_classifier import cluster, data, evaluate, model, visualize
from location_classifier.config import Config

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

## 1. Build a dataset

`make_synthetic_dataset` scatters points around real reference sites with a
Gaussian in the local north/east plane, so `spread_km` means the same thing at
every latitude.

Two built-in collections ship with the package:

* `metros` - eight Indian metros, hundreds of km apart and trivially separable.
* `zones` - eight neighbourhoods inside Mumbai, 5-15 km apart. With a realistic
  scatter these classes genuinely overlap, which is the interesting case.

Point `Config.data_path` at your own CSV to use real data instead; it only needs
`latitude`, `longitude` and `location` columns.

In [ ]:
frame = data.make_synthetic_dataset(
    samples_per_class=150,
    spread_km=2.5,
    sites=data.MUMBAI_ZONES,
    random_seed=42,
)

print(data.describe_dataset(frame))
frame.head()

In [ ]:
visualize.plot_locations(frame, title="Mumbai zones")
plt.show()

## 2. Feature engineering

Raw latitude/longitude is a poor model input: it is discontinuous at the
antimeridian and its scale changes with latitude. `GeoFeatureBuilder` expands a
coordinate pair into unit-sphere cartesian coordinates, cyclical encodings, and
great-circle distances to anchors learned from the training set.

In [ ]:
from location_classifier.features import GeoFeatureBuilder

builder = GeoFeatureBuilder(n_anchors=10, random_state=42).fit(
    frame.drop(columns=["location"])
)
matrix = builder.transform(frame.drop(columns=["location"]))

print(f"{matrix.shape[1]} features from {builder.n_features_in_} input columns")
print(list(builder.get_feature_names_out()))

## 3. Train a classifier

In [ ]:
config = Config(n_anchors=10, cv_folds=5, random_seed=42)
result = model.train_model(frame, config=config)

print(f"model:        {result.model_name}")
print(f"train / test: {result.n_train} / {result.n_test}")
print(f"cv accuracy:  {result.cv_mean:.4f}")
print()
print(evaluate.format_metrics(result.metrics, title="Held-out evaluation"))

## 4. Where does it go wrong?

The confusion matrix shows the errors concentrating between adjacent zones -
exactly where the point clouds overlap on the map above.

In [ ]:
visualize.plot_confusion_matrix(result.metrics, title="Zone confusion")
plt.show()

evaluate.per_class_frame(result.metrics)

In [ ]:
importances = model.feature_importances(result.pipeline, limit=15)
visualize.plot_feature_importances(importances)
plt.show()

## 5. Compare every registered model

`compare_models` trains each estimator in the registry on the same split, so the
numbers are directly comparable.

In [ ]:
leaderboard = model.compare_models(frame, config=config)
visualize.plot_model_comparison(leaderboard)
plt.show()

leaderboard

## 6. Predict new points

`points_frame` fills the auxiliary columns the model was trained on with their
training medians, so a bare coordinate pair is still a valid input.

In [ ]:
queries = pd.DataFrame(
    {
        "latitude": [18.9220, 19.0176, 19.2183, 19.1197],
        "longitude": [72.8347, 72.8562, 72.9781, 72.9050],
        "note": ["near Colaba", "Dadar", "Thane", "Powai"],
    }
)

points = model.points_frame(result.pipeline, queries["latitude"], queries["longitude"])
predictions = model.predict_locations(result.pipeline, points, top_k=3)

pd.concat(
    [
        queries["note"],
        predictions[
            ["predicted_location", "confidence", "rank_2_location", "rank_2_confidence"]
        ],
    ],
    axis=1,
)

## 7. Clustering without labels

Clustering runs in a metric space measured in kilometres, so a cluster radius
means the same thing anywhere on the globe. Purity compares each discovered
cluster against the true labels we deliberately withheld.

In [ ]:
clusters = cluster.cluster_locations(
    frame, method="kmeans", n_clusters=8, random_seed=42
)
print(clusters.describe())

In [ ]:
visualize.plot_clusters(
    frame, clusters.labels, centroids=clusters.centroids, title="k-means clusters"
)
plt.show()

In [ ]:
dbscan = cluster.cluster_locations(frame, method="dbscan", eps_km=1.2, min_samples=8)
print(dbscan.describe())

## 8. Persist the model

`save_model` writes a joblib bundle plus a sibling JSON file holding the metrics
and class list, so a saved model is self-describing.

In [ ]:
path = model.save_model(result, config.model_path)
pipeline, metadata = model.load_model(path)

print(f"loaded {metadata['model_name']} trained at {metadata['trained_at']}")
print(f"classes: {metadata['classes']}")

## Next steps

* Swap in a real CSV export and re-run from section 3.
* Everything shown here is also available from the command line:
  `python -m location_classifier train --plots`.